## Notebook17b

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

In [ ]:
import numpy as np
import polars as pl

from funs import *
from plotnine import *
from polars import col as c
theme_set(theme_minimal())

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

### Reading the Data

In [ ]:
rva = pl.read_csv(ub + "data/flightsrva_flights.csv.gz", null_values=["NA"])
rva = rva.drop(c.time_hour)
rva

**Research Questions**: How do delays *cascade* through the day at the Richmond airport? If you are flying to a popular location such as Atlanta and miss your flight, how long do you need to wait for the next flight (assuming there is another one)?

### Questions

1. The flight data has separate columns for the year, month, day, hour, and minute of each departure. Combine these into a single datetime column called `time` using `pl.datetime`. Then, filter the data to show only flights within a 24-hour window: from noon on March 11, 2019 to noon on March 12, 2019. Create a scatter plot of the departure delay (`dep_delay`) over time. Use `scale_x_date` to label the x-axis with one tick per hour, showing only the hour (use the format `"%H"`). What pattern do you notice about how delays change over the course of the day?

In [ ]:
(
    rva
    .with_columns(
        time = pl.datetime(c.year, c.month, c.day, c.hour, c.minute)
    )
    .filter(c.time >= pl.datetime(2019, 3, 11, 12, 0))
    .filter(c.time <= pl.datetime(2019, 3, 12, 12, 0))
    .pipe(ggplot, aes("time", "dep_delay"))
    + geom_point()
    + scale_x_date(date_breaks="1 hour", date_labels="%H")
)

2. Now let's measure delays more systematically across the entire dataset. Start by creating the `time` column as before. Then, truncate the time to 3-hour blocks (using `c.time.dt.truncate("3h")`) and save the result in a new column called `time_hour`. Group the data by `time_hour` and compute two aggregations: `delay_prop`, which is the proportion of flights with a departure delay greater than 30 minutes (hint: `(c.dep_delay > 30).mean()`), and `count`, which is the number of flights in each group (using `pl.len()`). Sort the result by `time_hour` and save it as `delay_per_3hour`.

In [ ]:
delay_per_3hour = (
    rva
    .with_columns(
        time = pl.datetime(c.year, c.month, c.day, c.hour, c.minute)
    )
    .with_columns(
        time_hour = c.time.dt.truncate("3h")
    )
    .group_by(c.time_hour)
    .agg(
        delay_prop = (c.dep_delay > 30).mean(),
        count = pl.len()
    )
    .sort(c.time_hour)
)
delay_per_3hour

3. We want to see if delays in one 3-hour window predict delays in the next. Using the `delay_per_3hour` dataset, create a new column called `delay_prop_prev` that contains the *previous* row's value of `delay_prop`. You can do this using the `.shift(1)` method, which shifts all values down by one row (the first row becomes null). Then, select just the Pearson correlation between `delay_prop` and `delay_prop_prev` using `pl.corr()`. Is the correlation positive? What might this tell you about how delays cascade?

In [ ]:
(
    delay_per_3hour
    .with_columns(
        delay_prop_prev = c.delay_prop.shift(1)
    )
    .select(
        pl.corr(c.delay_prop, c.delay_prop_prev)
    )
)

4. The correlation you computed in the previous question includes shifts that cross midnight boundaries — for example, comparing the last window of one day to the first window of the next. This doesn't make much sense because delays generally reset overnight. To fix this, first extract the date from `time_hour` into a new column called `day`. Then, use `.shift(1).over(c.day)` to shift `delay_prop` only *within* each day. Compute the same correlation as before. How does limiting the shift to within each day change the result? Is the correlation stronger or weaker?

In [ ]:
(
    delay_per_3hour
    .with_columns(day = c.time_hour.dt.date())
    .with_columns(
        delay_prop_prev = c.delay_prop.shift(1).over(c.day)
    )
    .select(
        pl.corr(c.delay_prop, c.delay_prop_prev)
    )
)

5. Three-hour windows are fairly coarse. Let's try a finer resolution. Repeat what you did in Question 2, but now truncate the time to 1-hour blocks instead of 3-hour blocks. Save the result as `delay_per_hour`. The structure of the code should be identical to Question 2, with only the truncation interval changed.

In [ ]:
delay_per_hour = (
    rva
    .with_columns(
        time = pl.datetime(c.year, c.month, c.day, c.hour, c.minute)
    )
    .with_columns(
        time_hour = c.time.dt.truncate("1h")
    )
    .group_by(c.time_hour)
    .agg(
        delay_prop = (c.dep_delay > 30).mean(),
        count = pl.len()
    )
    .sort(c.time_hour)
)
delay_per_hour

6. With hourly data, raw values can be noisy. To smooth things out, use a rolling mean with a window size of 3 (and `min_samples=1` so that edge rows are not dropped) applied within each day using `.over(c.day)`. Save this smoothed column as `delay_roll`. Then, shift the smoothed column by one row within each day to get `delay_roll_prev`. Finally, compute the correlation between the *raw* `delay_prop` and the *smoothed previous* value `delay_roll_prev`. How does this compare to the correlation you found with the 3-hour windows? The idea here is that the rolling average of previous delays may be a better predictor of the next hour's delays than just the single previous hour.

In [ ]:
(
    delay_per_hour
    .with_columns(day = c.time_hour.dt.date())
    .with_columns(
        delay_roll = c.delay_prop.rolling_mean(window_size=3, min_samples=1).over(c.day)
    )
    .with_columns(
        delay_roll_prev = c.delay_roll.shift(1).over(c.day)
    )
    .select(
        pl.corr(c.delay_prop, c.delay_roll_prev)
    )    
)

7. Now let's switch to the second research question. Suppose you are flying from Richmond to Atlanta (destination code `"ATL"`) and you miss your flight — how long until the next one? Start by creating the `time` column as before, then filter the data to only include flights with destination `"ATL"`. Sort the result by `time` and save it as `rva_atl`.

In [ ]:
rva_atl = (
    rva
    .with_columns(
        time = pl.datetime(c.year, c.month, c.day, c.hour, c.minute)
    )
    .filter(c.dest == "ATL")
    .sort(c.time)
)
rva_atl

8. One approach to finding the next flight is to use `join_where`, which joins two tables based on an inequality condition. Take `rva_atl` and add a column `time_2hr` that adds 2 hours to `time` (using `pl.duration(hours=2)`). Then use `join_where` to join `rva_atl` to itself, with the condition that the right table's `time` is strictly between the left table's `time` and `time_2hr`. This finds, for each flight, all other ATL flights departing within the next 2 hours. Select just the `time` column and call `.unique()` to see which departure times have at least one subsequent flight within 2 hours.

In [ ]:
(
    rva_atl
    .with_columns(
        time_2hr = c.time + pl.duration(hours=2)
    )
    .join_where(
        rva_atl,
        (c.time_right > c.time) & (c.time_right < c.time_2hr)
    )
    .select(c.time)
    .unique()
)

9. The `join_where` approach works but is somewhat cumbersome for this task. A cleaner approach is `join_asof`, which is designed for joining on ordered data like timestamps. It matches each row in the left table to the *nearest* row in the right table based on a key column. Here, we join `rva_atl` to itself on the `time` column. We need to set several options: `strategy="forward"` tells the join to look for the nearest *future* match (rather than the default, which looks backward); `on=c.time` specifies the column to match on (it must be sorted, which it is since we sorted `rva_atl` earlier); `allow_exact_matches=False` prevents a flight from matching with itself (since the same time would be an exact match); and `coalesce=False` keeps both the left and right versions of the join key so that we can see both departure times. Run the code and inspect the result — for each flight to Atlanta, you should see the next scheduled flight's information in the `_right` columns.

In [ ]:
(
    rva_atl
    .join_asof(
        rva_atl,
        strategy="forward",
        on=c.time,
        allow_exact_matches=False,
        coalesce=False
    )
)

10. Using the `join_asof` result from the previous question, filter the data so that the matched flight is on the *same day* (i.e., `c.day == c.day_right`) — we don't want to count waiting overnight as a valid option. Then, compute a new column `time_to_wait` as the difference between the next flight's time and the current flight's time (`c.time_right - c.time`). Create a histogram of `time_to_wait` using `geom_histogram` with `binwidth=0.01` and `fill="white"` and `color="black"`. What does the distribution of wait times look like? Are there common intervals between Atlanta flights from Richmond?

In [ ]:
(
    rva_atl
    .join_asof(
        rva_atl,
        strategy="forward",
        on=c.time,
        allow_exact_matches=False,
        coalesce=False
    )
    .filter(c.day == c.day_right)
    .with_columns(
        time_to_wait = c.time_right - c.time
    )
    .pipe(ggplot, aes("time_to_wait"))
    + geom_histogram(binwidth=0.01, fill="white", color="black")
)